# Phase 3: Model Training & Hyperparameter Tuning
This notebook covers:
1. Loading preprocessed data splits
2. Training multiple baseline classifiers (Logistic Regression, Naive Bayes, Random Forest)
3. 5-Fold Stratified Cross-Validation
4. GridSearch Hyperparameter Optimization
5. Saving best performing model pipeline

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from src.preprocessing import clean_data, build_preprocessor_pipeline, get_train_test_data

## 1. Prepare Data Splits

In [ ]:
df_raw = pd.read_csv('../data/raw/netflix_titles.csv')
df_clean = clean_data(df_raw)
X_train, X_test, y_train, y_test = get_train_test_data(df_clean)

## 2. Compare Multiple Classifiers (5-Fold Stratified Cross-Validation)

In [ ]:
preprocessor = build_preprocessor_pipeline()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')
    print(f"{name} 5-Fold Macro F1 Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 3. Hyperparameter Tuning using GridSearchCV

In [ ]:
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(full_pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Score (Macro F1): {grid_search.best_score_:.4f}")